# Appariage stable — qui peut tricher dans Gale–Shapley ?

**GameTheory-16i** (sous-famille *matching*) — **approfondissement** du [carnet 16](GameTheory-16-MechanismDesign-Python.ipynb) §5, qui y introduit déjà l'algorithme, la stabilité et l'avantage du côté proposeur.

L'**appariage stable** (*mariage stable*) de **Gale et Shapley (1962)** a valu à **Lloyd Shapley le prix Nobel d'économie 2012**. Le carnet 16 en pose les bases ; ici on pousse la **facette d'incitation** — la question que le §5 ne fait qu'évoquer à travers le théorème de **Gibbard–Satterthwaite** :

> Un agent peut-il améliorer sa situation en **mentant sur ses préférences** ?

Ce carnet **la mesure concrètement** : on implémente l'algorithme, on rappelle brièvement ses deux propriétés classiques (existence, avantage du proposeur), puis on montre par **recherche exhaustive** que **le côté proposeur est protégé** (aucun homme ne gagne à mentir) tandis que **le côté répondant peut tricher** (une femme s'améliore en déformant sa liste).

**Navigation** : [Index GameTheory](README.md) | Précédent : [GameTheory-16h — Coopération LLM](GameTheory-16h-Cooperation-LLM-Python.ipynb) | Base : [carnet 16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb).


In [1]:
import random
from itertools import permutations

random.seed(42)

def gale_shapley(proposers, prop_prefs, respondents, resp_prefs):
    """Algorithme de Gale-Shapley (recherche deferee / deferred acceptance).

    prop_prefs : dict  proposeur  -> [respondant, ...]  (du plus au moins preferé)
    resp_prefs : dict  respondent -> [proposeur,  ...]  (du plus au moins preferé)

    Retourne : dict  proposeur -> respondent  (l'appariage).
    """
    rank = {r: {p: k for k, p in enumerate(resp_prefs[r])} for r in respondents}
    libres = list(proposers)
    suivant = {p: 0 for p in proposers}
    appariage = {}   # proposeur  -> respondent
    actuel = {}      # respondent -> proposeur
    while libres:
        p = libres.pop()
        while suivant[p] < len(prop_prefs[p]):
            r = prop_prefs[p][suivant[p]]
            suivant[p] += 1
            if r not in actuel:
                actuel[r] = p
                appariage[p] = r
                break
            q = actuel[r]
            if rank[r][p] < rank[r][q]:      # r prefere p a son actuel q
                actuel[r] = p
                appariage[p] = r
                del appariage[q]
                libres.append(q)
                break
    return appariage

def trouver_paire_bloquante(appariage, prop_prefs, resp_prefs):
    """Retourne une paire bloquante (proposeur, respondent), ou None si stable."""
    rank = {r: {p: k for k, p in enumerate(resp_prefs[r])} for r in resp_prefs}
    actuel = {r: p for p, r in appariage.items()}
    for p, r in appariage.items():
        for r2 in prop_prefs[p]:
            if r2 == r:
                break                            # p ne prefere personne au-dessus de r
            if rank[r2][p] < rank[r2][actuel[r2]]:
                return (p, r2)                   # p prefere r2, et r2 prefere p a son actuel
    return None


## 1. Le problème de l'appariage stable

On a **n hommes** et **n femmes**, chacun avec une **liste de préférences** complète et strictement ordonnée sur l'autre sexe. Un **appariage** associe chaque homme à une femme (bijection).

Une **paire bloquante** est une paire (homme, femme) *non* appariés ensemble, où **les deux** préféreraient s'associer l'un à l'autre plutôt qu'à leur partenaire actuel. Un appariage est **stable** s'il n'admet **aucune** paire bloquante.

L'intuition : un appariage instable « s'effondre » — la paire bloquante s'unit et rompt les deux mariages existants. La question est : *existe-t-il toujours un appariage stable, et comment le trouver ?*

In [2]:
# Instance A (3 x 3) : une instance a DEUX appariages stables distincts
HOMMES = ["m1", "m2", "m3"]
FEMMES = ["w1", "w2", "w3"]

H_prefs = {
    "m1": ["w1", "w2", "w3"],
    "m2": ["w2", "w3", "w1"],
    "m3": ["w3", "w1", "w2"],
}
F_prefs = {
    "w1": ["m3", "m1", "m2"],
    "w2": ["m1", "m2", "m3"],
    "w3": ["m2", "m3", "m1"],
}

for h in HOMMES:
    print(f"{h} : {' > '.join(H_prefs[h])}")
print()
for f in FEMMES:
    print(f"{f} : {' > '.join(F_prefs[f])}")


m1 : w1 > w2 > w3
m2 : w2 > w3 > w1
m3 : w3 > w1 > w2

w1 : m3 > m1 > m2
w2 : m1 > m2 > m3
w3 : m2 > m3 > m1


## 2. L'algorithme de Gale–Shapley (*deferred acceptance*)

L'algorithme fait **proposer** à un côté (ici les hommes) :

1. un homme **libre** propose à la **première** femme de sa liste qu'il n'a pas encore sollicitée ;
2. si la femme est **libre**, elle accepte temporairement ;
3. si elle est **engagée**, elle garde le **préféré** entre son prétendant actuel et le nouveau, et libère l'autre ;
4. on répète jusqu'à ce que plus personne ne soit libre.

Chaque engagement est **réversible** (d'où *deferred acceptance*) : une femme peut « lâcher » son partenaire actuel pour un meilleur. L'algorithme **termine toujours** (chaque proposition élimine définitivement une paire homme-femme, et il en est un nombre fini) et produit un appariage **stable**.

In [3]:
# Résultat 1 : l'algorithme termine et produit un appariage STABLE
appariage = gale_shapley(HOMMES, H_prefs, FEMMES, F_prefs)
print("Appariage produit (hommes proposent) :", appariage)

pb = trouver_paire_bloquante(appariage, H_prefs, F_prefs)
print("Paire bloquante :", pb, "  ->  stable" if pb is None else "  ->  INSTABLE")


Appariage produit (hommes proposent) : {'m3': 'w3', 'm2': 'w2', 'm1': 'w1'}
Paire bloquante : None   ->  stable


### Résultat 1 — Existence

La mesure ci-dessus le confirme : l'algorithme **termine** et l'appariage produit ne contient **aucune paire bloquante** (`trouver_paire_bloquante` retourne `None`).

C'est le théorème d'**existence** de Gale–Shapley : *pour tout problème d'appariage stable, il existe au moins un appariage stable* — et l'algorithme le trouve en **temps polynomial** (O(n²) propositions au plus).

In [4]:
# Résultat 2 : les DEUX equilibres stables, et l'avantage du cote proposeur
m_opt = gale_shapley(HOMMES, H_prefs, FEMMES, F_prefs)   # hommes proposent
f_opt = gale_shapley(FEMMES, F_prefs, HOMMES, H_prefs)   # femmes proposent
f_opt = {f: h for h, f in f_opt.items()}                 # re-exprime en {homme: femme}

print("Equilibre 'hommes proposent' :", m_opt)
print("Equilibre 'femmes proposent' :", f_opt)
print("Identiques ?", m_opt == f_opt)
print()
print("Rang du partenaire de chaque homme (1 = son premier choix) :")
for h in HOMMES:
    r_h = H_prefs[h].index(m_opt[h]) + 1
    r_f = H_prefs[h].index(f_opt[h]) + 1
    print(f"  {h} : {r_h}e si les hommes proposent, {r_f}e si les femmes proposent")


Equilibre 'hommes proposent' : {'m3': 'w3', 'm2': 'w2', 'm1': 'w1'}
Equilibre 'femmes proposent' : {'m2': 'w3', 'm1': 'w2', 'm3': 'w1'}
Identiques ? False

Rang du partenaire de chaque homme (1 = son premier choix) :
  m1 : 1e si les hommes proposent, 2e si les femmes proposent
  m2 : 1e si les hommes proposent, 2e si les femmes proposent
  m3 : 1e si les hommes proposent, 2e si les femmes proposent


### Résultat 2 — L'avantage du côté proposeur

Il n'y a en général **pas un** mais **plusieurs** appariages stables : la mesure montre que l'équilibre « hommes proposent » et l'équilibre « femmes proposent » **diffèrent**.

Plus précisément, l'équilibre où les **hommes** proposent est **meilleur pour chaque homme** (et pire pour chaque femme) que l'équilibre où les **femmes** proposent : le côté **proposeur** obtient son **meilleur** appariage stable possible, le côté **répondant** son **pire**. C'est le théorème d'**optimalité** de Gale–Shapley — une asymétrie fondamentale : *dans la conception du mécanisme, donner le pouvoir de proposer avantage celui qui propose*.

In [5]:
# Instance C (3 x 3) : le cote proposeur est protege, le cote repondant non
C_H = {
    "m1": ["w2", "w1", "w3"],
    "m2": ["w2", "w3", "w1"],
    "m3": ["w1", "w2", "w3"],
}
C_F = {
    "w1": ["m2", "m1", "m3"],
    "w2": ["m3", "m1", "m2"],
    "w3": ["m1", "m2", "m3"],
}

C_honnete = gale_shapley(list(C_H), C_H, list(C_F), C_F)
print("Appariage honnete (hommes proposent) :", C_honnete)

def rang_m(h, f):   return C_H[h].index(f) + 1   # rang d'une femme dans la liste d'un homme
def rang_f(f, h):   return C_F[f].index(h) + 1   # rang d'un homme  dans la liste d'une femme

# (A) Un HOMME peut-il s'ameliorer en mentant ? Recherche EXHAUSTIVE :
# pour chaque homme, on essaie TOUTES les permutations de sa liste.
hommes = 0
for h in C_H:
    base = rang_m(h, C_honnete[h])
    for perm in permutations(C_H[h]):
        p2 = dict(C_H); p2[h] = list(perm)
        m2 = gale_shapley(list(C_H), p2, list(C_F), C_F)
        if rang_m(h, m2[h]) < base:               # strictement meilleur (rang plus petit)
            hommes += 1
            break
print("\n(A) Hommes pouvant s'ameliorer en mentant :", hommes, "sur", len(C_H))


Appariage honnete (hommes proposent) : {'m3': 'w1', 'm1': 'w2', 'm2': 'w3'}

(A) Hommes pouvant s'ameliorer en mentant : 0 sur 3


In [6]:
# (B) Une FEMME peut-elle s'ameliorer en mentant ? Recherche EXHAUSTIVE.
part = {f: h for h, f in C_honnete.items()}
trouve = None
for f in C_F:
    base = rang_f(f, part[f])
    for perm in permutations(C_F[f]):
        fp2 = dict(C_F); fp2[f] = list(perm)
        m2 = gale_shapley(list(C_H), C_H, list(C_F), fp2)
        p2 = {w: h for h, w in m2.items()}
        if rang_f(f, p2[f]) < base:               # un homme qu'elle prefere vraiment
            trouve = (f, list(perm), part[f], base, p2[f], rang_f(f, p2[f]))
            break
    if trouve:
        break

if trouve:
    f, decl, h0, r0, h1, r1 = trouve
    print(f"(B) {f} (liste vraie : {C_F[f]})")
    print(f"    honnête : {h0} (son rang {r0})  ->  declare {decl}  ->  {h1} (son rang {r1}) : MEILLEUR")
else:
    print("(B) aucune femme ne peut s'ameliorer ici.")


(B) w2 (liste vraie : ['m3', 'm1', 'm2'])
    honnête : m1 (son rang 2)  ->  declare ['m3', 'm2', 'm1']  ->  m3 (son rang 1) : MEILLEUR


### Résultat 3 — La non-stratégicité : qui peut tricher ?

La mesure **(A)** est surprenante : **aucun homme** ne peut améliorer son appariage en mentant sur ses préférences. C'est un **théorème** : dans le Gale–Shapley où les hommes proposent, l'issue honnête de chaque homme est déjà sa **meilleure** issue stable possible — mentir ne peut que le **nuire**, jamais l'avantager. Le **côté proposeur est protégé**.

La mesure **(B)** révèle l'asymétrie : la **femme w2** obtient un **meilleur** partenaire (son rang 1, m3) en déplaçant simplement m1 en bas de sa liste. Le **côté répondant n'est pas protégé** — il peut exploiter le mécanisme. L'algorithme n'est donc pas *sans stratagème* (*not strategy-proof*), mais **uniquement du côté des répondants**.

D'où le principe de conception : **donner le pouvoir de proposer au côté qu'on veut protéger**. C'est exactement ce que fait le matching médical : les **résidents** (médecins en formation) sont le côté proposeur, précisément parce que l'algorithme les protège de leur propre manipulation stratégique, même s'il ne protège pas les hôpitaux.

## 4. Exercices

In [7]:
# Exercice 1
# TODO etudiant : completer la boucle de recherche deferee (deferred acceptance).
# A chaque tour, un homme libre propose a la premiere femme de sa liste qu'il n'a
# pas encore sollicitee ; une femme engagee garde son pretendu qu'elle prefere.
# # Indice : garder `libres` (liste), `actuel` (femme -> homme) et `suivant`
# #         (homme -> indice dans sa liste) ; l'algorithme du carnet est le modele.
def gale_shapley_etudiant(homes, homes_prefs, femmes, femmes_prefs):
    # TODO etudiant
    return None  # TODO etudiant : retourner l'appariage {homme: femme}


In [8]:
# Exercice 2
# TODO etudiant : ecrire un verificateur de stabilite.
# # Indice : pour chaque homme p apparie a r, parcourez la liste de p AU-DESSUS de r ;
# #         si une telle femme r2 prefere p a son partenaire actuel, c'est une paire bloquante.
def trouver_paire_bloquante_etudiant(appariage, homes_prefs, femmes_prefs):
    # TODO etudiant
    return None  # TODO etudiant : la paire bloquante (homme, femme), ou None si stable


In [9]:
# Exercice 3
# TODO etudiant : le cote PROPOSEUR (hommes) est protege, mais le cote REPONDANT (femmes)
# non. Pour une femme donnee, trouver une declaration mensongere (une permutation de sa
# liste de preferences) qui lui donne un partenaire strictement MEILLEUR (un rang plus
# petit dans SA vraie liste) que le jeu honnete.
# # Indice : iterez sur `permutations(liste_femme)`, relancez gale_shapley en ne modifiant
# #         que la liste de cette femme, et comparez le rang (dans sa vraie liste) du
# #         partenaire obtenu au rang honnete.
def declaration_profitable_etudiant(femme, homes_prefs, femmes_prefs):
    # TODO etudiant
    return None  # TODO etudiant : la declaration profitable, ou None s'il n'y en a pas


## En synthèse

L'algorithme de Gale–Shapley transforme un problème à l'apparence instable en un **appariage stable garanti**, en temps polynomial. Ses trois propriétés mesurées ici — **existence**, **avantage du proposeur**, **non-stratégicité** — en font à la fois un outil pratiquement invaluable et un objet théoriquement riche : elles encodent le fait que *la règle du jeu (qui propose) distribue les gains*. Les versions modernes (matching médical, allocation de logements) en héritent directement.